# 4장 도메인 주도 설계: 핵심 비즈니스 로직 설계

파이썬으로 구현하는 클린 아키텍처 - 4장 도메인 주도 설계: 핵심 비즈니스 로직 설계 코드 예제

> **[노트북 참고]** 아래 셀은 노트북 환경에서 `TodoApp` 코드를 import할 수 있도록 경로를 설정합니다. 반드시 첫 번째로 실행해 주세요.

In [ ]:
# ============================================================
# [추가] 노트북 환경 설정
# TodoApp 패키지를 import하기 위한 경로 설정 (Colab/로컬 환경 자동 감지)
# 반드시 첫 번째로 실행해 주세요.
# ============================================================
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Colab 환경: GitHub에서 코드를 클론하여 경로 설정
if IN_COLAB:
    if not os.path.exists('/content/repo'):
        !git clone https://github.com/songys/Clean-Architecture-with-Python.git /content/repo
    TODOAPP_PATH = '/content/repo/Chapter_4/TodoApp'
# 로컬 환경: 현재 디렉토리의 TodoApp 경로 설정
else:
    TODOAPP_PATH = os.path.join(os.getcwd(), 'TodoApp')

if TODOAPP_PATH not in sys.path:
    sys.path.insert(0, TODOAPP_PATH)

## 개요

이 장에서는 도메인 주도 설계(DDD) 원칙에 따라 엔터티 계층을 구현하는 방법을 깊이 있게 다룬다.

이 장에서 다루는 주요 주제:
* DDD 원칙에 따른 핵심 엔터티 식별과 모델링
* 파이썬으로 엔터티 구현
* 고급 도메인 개념

### Entity 기본 클래스

## 엔터티의 토대: 식별성과 동등성

모든 엔터티의 기본 클래스로, **고유 식별자(UUID)**를 부여하고 동등성 비교(`__eq__`)와 해싱(`__hash__`)이 올바르게 동작하도록 한다. 속성이 같더라도 ID가 다른 두 엔터티는 서로 다른 객체로 간주된다.

In [ ]:
# Entity 기본 클래스 - 모든 엔터티의 공통 토대
# DDD에서 엔터티는 고유 식별자(UUID)로 구별되는 도메인 객체
from dataclasses import dataclass, field
from uuid import UUID, uuid4


@dataclass
class Entity:
    # UUID 자동 생성 - 각 엔터티에 고유한 식별자 부여 (init=False로 외부 설정 불가)
    id: UUID = field(default_factory=uuid4, init=False)

    # 동등성 비교: 속성이 아닌 ID로만 판단 (엔터티의 핵심 특성)
    def __eq__(self, other: object) -> bool:
        if not isinstance(other, type(self)):
            return NotImplemented
        return self.id == other.id

    # 해시 함수: ID 기반으로 set, dict의 키로 사용 가능
    def __hash__(self) -> int:
        return hash(self.id)

### 값 객체 정의 (TaskStatus, Priority, Deadline)

## 데이터 무결성과 명확한 의미 부여

값 객체는 **속성 값으로 동등성을 판단**하고 **불변(immutable)**이다. `Deadline`은 `frozen=True` 데이터클래스로 구현되어 생성 후 변경이 불가능하며, `__post_init__`에서 과거 날짜를 거부하는 **자체 유효성 검증**을 수행한다. `is_overdue()`, `time_remaining()`, `is_approaching()` 등 도메인 로직을 캡슐화한다.

In [ ]:
# 값 객체(Value Object) 정의 - 불변성과 자체 유효성 검증이 핵심 특성
from enum import Enum
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone


# 작업 상태 열거형 - TODO → IN_PROGRESS → DONE 순서로 상태 전이
class TaskStatus(Enum):
    TODO = "TODO"
    IN_PROGRESS = "IN_PROGRESS"
    DONE = "DONE"


# 우선순위 열거형 - 숫자 값(1, 2, 3)을 통한 우선순위 간 비교 가능
class Priority(Enum):
    LOW = 1
    MEDIUM = 2
    HIGH = 3


# 마감일 값 객체 - frozen=True로 불변(immutable) 보장
# 값 객체의 핵심 특성: 불변성, 속성 값 기반 동등성, 자체 유효성 검증
@dataclass(frozen=True)
class Deadline:
    due_date: datetime

    # 생성 시 자동 유효성 검증: 과거 날짜 거부
    def __post_init__(self):
        if self.due_date < datetime.now(timezone.utc):
            raise ValueError("Deadline cannot be in the past")

    # 마감일 초과 여부 확인
    def is_overdue(self) -> bool:
        return datetime.now(timezone.utc) > self.due_date

    # 남은 시간 계산 - 음수 방지를 위해 최솟값 0으로 제한
    def time_remaining(self) -> timedelta:
        return max(
            timedelta(0),
            self.due_date - datetime.now(timezone.utc)
        )

    # 마감일 임박 여부 확인 - 기본 경고 기준: 1일
    def is_approaching(
        self, warning_threshold: timedelta = timedelta(days=1)
    ) -> bool:
        return timedelta(0) < self.time_remaining() <= warning_threshold

### 00_create_new_task.py

## 작업(Task) 엔터티 생성

도메인 주도 설계(DDD)에서 **엔터티**는 고유한 식별자를 가지며 시간에 따라 상태가 변하는 도메인 객체이다. 이 코드는 Task 엔터티의 생성과 기본 구조를 보여준다.

`@dataclass`의 `frozen=True`를 사용하면 불변(immutable) 객체를 만들 수 있어 값 객체(Value Object)에 적합하다.

In [ ]:
# Task 엔터티 생성 예제 - DDD에서 엔터티는 고유 식별자를 가진 도메인 객체
from todo_app.domain.entities.task import Task
from todo_app.domain.value_objects import Priority

# 값 객체(Priority.HIGH)를 사용하여 새 작업 생성
task = Task(
    title="Complete project proposal",
    description="Draft and review the proposal for the new client project",
    priority=Priority.HIGH,
)

# 작업 속성 확인 - status는 생성 시 기본값 TaskStatus.TODO로 자동 설정
print(task.title)  # "Complete project proposal"
print(task.priority)  # Priority.HIGH
print(task.status)  # TaskStatus.TODO

### 01_create_task_business_rules.py

## 비즈니스 규칙이 포함된 작업 엔터티

엔터티는 단순한 데이터 컨테이너가 아니라, **비즈니스 규칙을 캡슐화**하는 풍부한 도메인 모델이다. 작업 상태 전이, 기한 확인 등의 비즈니스 로직이 엔터티 내부에 위치한다.

이 방식은 '빈약한 도메인 모델(Anemic Domain Model)' 안티 패턴을 피하고, 비즈니스 규칙을 도메인 계층에 집중시킨다.

In [ ]:
# 비즈니스 규칙이 포함된 Task 엔터티 사용 예제
# 엔터티는 단순 데이터 컨테이너가 아닌, 비즈니스 규칙을 캡슐화하는 풍부한 도메인 모델
from datetime import datetime, timedelta, timezone  # [수정] timezone 추가

from todo_app.domain.entities.task import Task
from todo_app.domain.value_objects import Deadline, Priority

# 마감일과 우선순위를 포함한 작업 생성
task = Task(
    title="Complete project proposal",
    description="Draft and review the proposal for the new client project",
    due_date=Deadline(datetime.now(timezone.utc) + timedelta(days=7)),  # [수정] timezone.utc 추가
    priority=Priority.HIGH,
)

# 작업 시작 - 상태 전이: TODO → IN_PROGRESS
task.start()
print(task.status)  # TaskStatus.IN_PROGRESS

# 작업 완료 - 상태 전이: IN_PROGRESS → DONE
task.complete()
print(task.status)  # TaskStatus.DONE

# 비즈니스 규칙 검증: 완료된 작업은 다시 시작 불가
try:
    task.start()  # ValueError가 발생함
except ValueError as e:
    print(str(e))  # "'TODO' 상태인 작업만 시작 가능"

# 마감일 초과 여부 확인 - Deadline 값 객체의 도메인 로직 활용
print(task.is_overdue())  # False

### 02_value_objects_in_clean_arch.py

## 값 객체(Value Objects)

값 객체는 식별자가 없고 속성 값으로만 동등성이 결정되는 도메인 객체이다. `TaskStatus` 열거형이나 `Priority` 같은 타입이 값 객체의 예시이다.

값 객체의 특성:
- 불변(immutable)
- 속성 값으로 동등성 판단
- 자체 유효성 검증 로직 포함
- 도메인 개념을 명시적으로 표현

In [ ]:
# 값 객체(Value Object) vs 원시 타입(문자열) 비교 예제
# 문자열 사용의 문제점과 열거형(Enum) 사용의 장점을 대비하여 설명
from todo_app.domain.entities.task import Task
from todo_app.domain.value_objects import TaskStatus

# 문자열 사용 (문제 있음) - 오타, 대소문자 불일치 등 런타임 오류 가능성
task = Task("Complete project", "The important project")
task.status = "Finished"  # 허용되지만 유효하지 않음 (정의되지 않은 상태값)
print(task.status == "done")  # False, 대소문자 구분으로 인한 비교 실패

# TaskStatus 열거형 사용 (견고함) - 타입 안전성과 유효한 값만 허용
task = Task("Complete project", "The important project")
task.status = TaskStatus.DONE  # 타입 안전 - 정의된 값만 사용 가능
print(task.status == TaskStatus.DONE)  # True, 대소문자 문제 없음

### 도메인 서비스: TaskPriorityCalculator

## DDD의 3대 핵심 구성 요소: 엔터티, 값 객체, 도메인 서비스

특정 엔터티에 속하지 않는 **상태 없는(stateless) 연산**은 도메인 서비스로 분리한다. `TaskPriorityCalculator`는 마감일을 기준으로 작업 우선순위를 계산하는 로직을 캡슐화한다. 여러 엔터티에 걸친 도메인 로직을 처리하면서도 도메인 계층 내에 위치한다.

In [ ]:
# 도메인 서비스 - 특정 엔터티에 속하지 않는 상태 없는(stateless) 연산
# 여러 엔터티에 걸친 도메인 로직을 처리하면서 도메인 계층 내에 위치
from datetime import timedelta


# TaskPriorityCalculator: 마감일을 기준으로 작업 우선순위를 계산하는 도메인 서비스
# @staticmethod로 상태 없는 연산임을 명시
class TaskPriorityCalculator:
    @staticmethod
    def calculate_priority(task: Task) -> Priority:
        # 마감일 초과 시 최우선 처리
        if task.is_overdue():
            return Priority.HIGH
        # 마감일 2일 이내 접근 시 중간 우선순위
        elif (
            task.due_date and task.due_date.time_remaining() <=
            timedelta(days=2)
        ):
            return Priority.MEDIUM
        # 그 외 기본 낮은 우선순위
        else:
            return Priority.LOW

### 03_project_usage.py

## 애그리게이트(Aggregate) 패턴: 프로젝트와 작업

`Project`는 여러 `Task`를 관리하는 **애그리게이트 루트(Aggregate Root)**이다. 애그리게이트는 DDD의 핵심 패턴으로 다음과 같은 특징을 가진다:

- **캡슐화**: 작업 추가/제거가 반드시 프로젝트를 통해서만 이루어짐
- **일관성**: 프로젝트 내 작업들의 비즈니스 규칙이 항상 유지됨
- **트랜잭션 경계**: 프로젝트 단위로 데이터 변경이 원자적으로 처리됨
- **불변 조건**: 프로젝트의 상태가 항상 유효한 상태를 유지

In [ ]:
# 애그리게이트(Aggregate) 패턴 사용 예제
# Project는 여러 Task를 관리하는 애그리게이트 루트(Aggregate Root)
# 작업 추가/제거가 반드시 프로젝트를 통해서만 이루어지는 캡슐화
from datetime import datetime, timedelta, timezone  # [수정] timedelta, timezone 추가

from todo_app.domain.entities.project import Project
from todo_app.domain.entities.task import Task
from todo_app.domain.value_objects import Deadline, Priority

# 프로젝트(애그리게이트 루트) 생성
project = Project("Website Redesign")
# 개별 작업(엔터티) 생성 후 프로젝트에 추가
task1 = Task(
    title="Design homepage",
    description="Create new homepage layout",
    due_date=Deadline(datetime.now(timezone.utc) + timedelta(days=30)),  # [수정] 미래 날짜 + timezone
    priority=Priority.HIGH,
)
task2 = Task(
    title="Implement login",
    description="Add user authentication",
    due_date=Deadline(datetime.now(timezone.utc) + timedelta(days=15)),  # [수정] 미래 날짜 + timezone
    priority=Priority.MEDIUM,
)
# 애그리게이트 루트를 통한 작업 추가 - 프로젝트의 일관성 규칙 보장
project.add_task(task1)
project.add_task(task2)

print(f"프로젝트: {project.name}")
print(f"작업 수: {len(project.tasks)}")
print(f"첫 번째 작업: {project.tasks[0].title}")

### 04_factory_pattern_class_methods.py

## 팩토리 패턴 - 클래스 메서드

`@classmethod`를 팩토리 메서드로 활용하여 특정 비즈니스 규칙을 내장한 객체 생성 인터페이스를 제공한다. 예: `create_urgent_task()`는 `Priority.HIGH`를 자동 설정한다.

In [ ]:
# 팩토리 패턴 - 클래스 메서드(@classmethod) 활용
# 복잡한 객체 생성 로직을 캡슐화하여 일관된 생성 인터페이스 제공
from dataclasses import dataclass, field
from typing import Optional

from todo_app.domain.entities.entity import Entity
from todo_app.domain.value_objects import Priority, Deadline, TaskStatus


@dataclass
class Task(Entity):
    # [보완] 기존 속성들을 실제 소스에서 가져옴
    title: str = ""
    description: str = ""
    due_date: Optional[Deadline] = None
    priority: Priority = Priority.MEDIUM
    status: TaskStatus = field(default=TaskStatus.TODO, init=False)

    # 팩토리 메서드: 긴급 작업 생성 시 Priority.HIGH를 자동 설정하는 편의 메서드
    # 일반 생성자와 달리 특정 비즈니스 규칙을 내장한 생성 로직
    @classmethod
    def create_urgent_task(cls, title: str, description: str, due_date: Deadline):
        return cls(title, description, due_date, Priority.HIGH)

### 05_factory_pattern_post_init.py

## 팩토리 패턴 - `__post_init__`

`dataclass`의 `__post_init__` 메서드를 활용하여 객체 생성 직후 유효성 검증이나 파생 속성 계산을 수행한다.

In [ ]:
# 팩토리 패턴 - __post_init__ 활용
# dataclass의 __post_init__으로 객체 생성 직후 유효성 검증 수행
from dataclasses import dataclass, field
from typing import Optional

from todo_app.domain.entities.entity import Entity
from todo_app.domain.value_objects import Priority, Deadline, TaskStatus


@dataclass
class Task(Entity):
    # [보완] 기존 속성들을 실제 소스에서 가져옴
    title: str = ""
    description: str = ""
    due_date: Optional[Deadline] = None
    priority: Priority = Priority.MEDIUM
    status: TaskStatus = field(default=TaskStatus.TODO, init=False)

    # 생성 직후 자동 실행되는 유효성 검증 메서드
    # 비즈니스 규칙: 빈 제목 불가, 설명 500자 제한
    def __post_init__(self):
        if not self.title.strip():
            raise ValueError("비어 있는 작업 제목 사용 불가능")
        if len(self.description) > 500:
            raise ValueError(
                "작업 설명 500자 초과 불가")

### 06_factory_pattern_task_factory.py

## 작업 팩토리(Task Factory)

프로젝트 상태나 담당자 역할에 따라 달라지는 복잡한 생성 로직은 독립적인 팩토리 클래스로 분리한다. `@classmethod`나 `__post_init__`으로 처리하기 어려운 **여러 엔터티에 걸친 비즈니스 규칙**이 있을 때 적합하다.

In [ ]:
# 독립적인 팩토리 클래스 - 복잡한 객체 생성 로직의 캡슐화
# 프로젝트 상태와 사용자 권한에 따라 달라지는 비즈니스 규칙을 포함한 생성 로직
from uuid import UUID

from todo_app.domain.entities.task import Task
from todo_app.domain.value_objects import Priority


class TaskFactory:
    # 의존성 주입: 사용자 서비스와 프로젝트 리포지토리를 외부에서 전달
    def __init__(self, user_service, project_repository):
        self.user_service = user_service
        self.project_repository = project_repository

    # 프로젝트 맥락에서 작업을 생성하는 팩토리 메서드
    # 프로젝트 우선순위와 담당자 역할에 따른 비즈니스 규칙 적용
    def create_task_in_project(
        self, title: str, description: str, project_id: UUID, assignee_id: UUID
    ):
        project = self.project_repository.get_by_id(project_id)
        assignee = self.user_service.get_user(assignee_id)

        task = Task(title, description)
        task.project = project
        task.assignee = assignee

        # 비즈니스 규칙: 고우선 프로젝트의 관리자 작업은 자동으로 HIGH 우선순위
        if project.is_high_priority() and assignee.is_manager():
            task.priority = Priority.HIGH

        # 애그리게이트 루트를 통한 작업 추가
        project.add_task(task)
        return task

### 07_dependency_rule_broken.py

## 의존성 규칙 위반 사례

도메인 엔터티가 DB 연결이나 UI 컴포넌트에 직접 의존하면 의존성 규칙을 위반한다. 아래 두 가지 안티패턴을 통해 위반 사례와 해결 방향을 살펴본다.

In [ ]:
# 의존성 규칙 위반 사례 모음 - 도메인 엔터티가 외부 관심사에 의존하는 안티패턴
from dataclasses import dataclass, field
from typing import Optional
from uuid import UUID

from todo_app.domain.entities.entity import Entity
from todo_app.domain.entities.task import Task
from todo_app.domain.value_objects import (
    TaskStatus,
    Deadline,
    Priority,
)


# 외부 인프라 클래스 (DB, UI) - 도메인 계층에서 직접 참조하면 안 되는 대상
class DbConnection:
    pass


class UiComponent:
    pass


# 위반 사례 1: 도메인 엔터티가 DB 연결에 직접 의존
# → 도메인 계층이 인프라 계층에 의존하여 의존성 규칙 위반
@dataclass
class TaskWithDatabase:
    """
    DB 연결을 통한 의존성 규칙 위반
    """

    title: str
    description: str
    db: DbConnection  # 의존성 규칙 위반 - 도메인이 인프라에 직접 의존
    due_date: Optional[Deadline] = None
    priority: Priority = Priority.MEDIUM
    status: TaskStatus = field(default=TaskStatus.TODO, init=False)

    def mark_as_complete(self):
        self.status = TaskStatus.DONE
        self.db.update(self)  # 도메인 로직에 인프라 호출이 혼합


# 위반 사례 2: 애그리게이트가 UI 컴포넌트에 직접 의존
# → 도메인 계층이 프레젠테이션 계층에 의존하여 의존성 규칙 위반
@dataclass
class ProjectWithUI(Entity):
    """
    UI 관심사를 통한 의존성 규칙 위반
    """

    name: str
    ui: UiComponent  # 의존성 규칙 위반 - 도메인이 UI에 직접 의존
    description: str = ""
    _tasks: dict[UUID, Task] = field(default_factory=dict, init=False)

    def add_task(self, task: Task):
        self._tasks[task.id] = task
        self.ui.refresh()  # 의존성 규칙 위반 - 도메인 로직에 UI 호출이 혼합

### 08_external_dependencies.py

## 외부 의존성 처리

도메인 계층의 순수성을 유지하려면 외부 관심사에 대한 의존을 **추상 인터페이스(포트)**로 분리한다. 도메인 계층에 리포지토리 인터페이스를 정의하고, 인프라 계층에서 구현하는 구조이다.

In [ ]:
# 외부 의존성 처리 - 추상 인터페이스를 통한 의존성 역전 원칙(DIP) 적용
# 도메인 계층에서 인터페이스를 정의하고, 인프라 계층에서 구현하는 구조

# ── 도메인 계층: 리포지토리 인터페이스 정의 (내부 원) ──
# 구현 세부사항 없이 작업 저장에 대한 계약만 정의
from abc import ABC, abstractmethod

from todo_app.domain.entities.task import Task


class TaskRepository(ABC):
    @abstractmethod
    def save(self, task: Task):
        pass

    @abstractmethod
    def get(self, task_id: str) -> Task:
        pass


# ── 도메인 서비스: 리포지토리 인터페이스에 의존 (내부 원) ──
# 구체적인 DB 구현이 아닌 추상 인터페이스에 의존하여 의존성 역전
class TaskService:
    def __init__(self, task_repository: TaskRepository):
        # 의존성 주입: 외부에서 구체적 리포지토리 구현체를 전달
        self.task_repository = task_repository

    def create_task(self, title: str, description: str) -> Task:
        task = Task(title, description)
        self.task_repository.save(task)
        return task

    def mark_task_as_complete(self, task_id: str) -> Task:
        task = self.task_repository.get(task_id)
        task.complete()
        self.task_repository.save(task)
        return task


# ── 인프라 계층: 구체적 구현체 (외부 원) ──
# 추상 인터페이스를 구현하는 SQLite 리포지토리 어댑터
class SQLiteTaskRepository(TaskRepository):
    def __init__(self, db_connection):
        self.db = db_connection

    def save(self, task: Task):
        # 구현 세부 사항...
        pass

    def get(self, task_id: str) -> Task:
        # 구현 세부 사항...
        pass

### 09_refactoring_before.py

## 리팩토링 전

도메인 로직과 인프라 관심사가 혼합된 코드이다. 비즈니스 규칙이 데이터베이스 접근 코드와 뒤섞여 있어 테스트와 유지보수가 어렵다.

In [ ]:
# 리팩토링 전 - 도메인 로직과 인프라 관심사가 혼합된 코드
# 비즈니스 규칙(상태 변경)과 인프라 로직(이메일 전송)이 뒤섞여 있는 안티패턴
from dataclasses import dataclass, field
from typing import Optional

from todo_app.domain.entities.entity import Entity
from todo_app.domain.value_objects import (
    Deadline,
    Priority,
    TaskStatus,
)


# 리팩토링 전 - 도메인 순수성 위반 사례
@dataclass
class Task(Entity):
    title: str
    description: str
    due_date: Optional[Deadline] = None
    priority: Priority = Priority.MEDIUM
    status: TaskStatus = field(default=TaskStatus.TODO, init=False)

    def mark_as_complete(self):
        self.status = TaskStatus.DONE
        # 이메일 알림 전송 - 도메인 순수성 위반 (인프라 관심사가 도메인에 침투)
        self.send_completion_email()

    # 이메일 전송 로직이 도메인 엔터티에 위치 - 분리 필요 대상
    def send_completion_email(self):
        # 이메일 알림을 전송하는 코드
        print(f"이메일 전송: 작업 '{self.title}' 완료")

### 10_refactoring_after.py

## 리팩토링 후: 알림 책임 분리

이메일 전송 책임을 도메인에서 분리하고, `TaskCompleteNotifier` 추상 인터페이스를 도입한다. **의존성 역전 원칙**에 따라 인프라 세부사항은 추상 인터페이스 뒤에 숨겨지고, 도메인 엔터티는 순수한 비즈니스 로직에만 집중한다.

In [ ]:
# 리팩토링 후 - 알림 책임을 도메인에서 분리
# 의존성 역전 원칙(DIP): 추상 인터페이스로 인프라 세부사항을 숨김
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional

from todo_app.domain.entities.entity import Entity
from todo_app.domain.value_objects import (
    Deadline,
    Priority,
    TaskStatus,
)


# 리팩토링 후 - 순수한 도메인 엔터티 (인프라 의존성 제거)
@dataclass
class Task(Entity):
    title: str
    description: str
    due_date: Optional[Deadline] = None
    priority: Priority = Priority.MEDIUM
    status: TaskStatus = field(default=TaskStatus.TODO, init=False)

    def mark_as_complete(self):
        self.status = TaskStatus.DONE
        # 여기서는 이메일을 전송하지 않음
        # 이제 이 책임은 외부 계층에 있음


# 알림 추상 인터페이스 - 도메인 계층에서 정의하는 포트
class TaskCompleteNotifier(ABC):
    @abstractmethod
    def notify_completion(self, task):
        pass


# 구체적 구현체 - 인프라 계층에서 추상 인터페이스를 구현하는 어댑터
class EmailTaskCompleteNotifier(TaskCompleteNotifier):
    def notify_completion(self, task):
        print(f"이메일 전송: 작업 '{task.title}' 완료")